In [1]:
import sys
import os

print("--- Path Analysis and Memory Purge ---")

# --- Step 1: Fix the Search Path ---
incorrect_path = '/home/lwan/.local/lib/python3.10/site-packages'
if incorrect_path in sys.path:
    print(f"Found and removed incorrect path: {incorrect_path}")
    while incorrect_path in sys.path:
        sys.path.remove(incorrect_path)
else:
    print("Incorrect path not found in search path.")

# --- Step 2: Purge All NumPy Modules from Memory ---
numpy_modules = []
for module in sys.modules:
    if module == 'numpy' or module.startswith('numpy.'):
        numpy_modules.append(module)

if numpy_modules:
    print(f"Found and purged {len(numpy_modules)} cached NumPy modules.")
    for module in numpy_modules:
        del sys.modules[module]
else:
    print("No cached NumPy modules found to purge.")

# --- Step 3: Verification ---
try:
    import numpy
    print("\n--- Verification Complete ---")
    print(f"Current NumPy Version: {numpy.__version__}")
    print(f"Current NumPy Location: {numpy.__file__}")

    correct_path_prefix = '/exp/dune/data/users/ayankele/transformercvn_env'
    if correct_path_prefix in numpy.__file__:
        print("\n✅ Success! The kernel is now using the correct NumPy.")
    else:
        print("\n❌ Error! The kernel is still using an incorrect NumPy.")

except ImportError as e:
    print(f"\n❌ An ImportError occurred after cleaning: {e}")

import numpy as np
import h5py
import os
import sys

from matplotlib import pyplot as plt
from tqdm import tqdm

import torch
import sparse
import numpy
import sys


--- Path Analysis and Memory Purge ---
Found and removed incorrect path: /home/lwan/.local/lib/python3.10/site-packages
No cached NumPy modules found to purge.

--- Verification Complete ---
Current NumPy Version: 1.23.5
Current NumPy Location: /home/lwan/transformerCVN-env/lib/python3.10/site-packages/numpy/__init__.py

❌ Error! The kernel is still using an incorrect NumPy.


In [2]:
import numba

@numba.njit()
def sparse_to_sparse(coords, values, num_prongs, num_features, shape_scale):
    DIMS = 4
    max_items = values.shape[0] + num_prongs
    
    output_coordinates = np.zeros((max_items, DIMS - 1), dtype=np.int64)
    output_values = np.zeros((max_items, num_features), dtype=np.float32)
    
    index_mapping = dict()
    num_unique_indices = 0

    for i in range(values.shape[0]): 
        v = values[i]
        if v == 0:
            continue

        b = coords[i, 0]
        c = coords[i, 1]
        y = coords[i, 2]
        x = coords[i, 3]
        
        if num_features == 3 and c == 1:
            y = 349 - y  ##Check this shape!!

        flat_index = shape_scale[0] * b + shape_scale[1] * y + shape_scale[2] * x

        if flat_index not in index_mapping:
            output_index = num_unique_indices
            index_mapping[flat_index] = num_unique_indices
            num_unique_indices += 1
        else:
            output_index = index_mapping[flat_index]

        output_coordinates[output_index][0] = b
        output_coordinates[output_index][1] = y
        output_coordinates[output_index][2] = x

        output_values[output_index][c] = v

    non_zero_planes = set(output_coordinates[:num_unique_indices, 0])
    all_planes = set(np.arange(num_prongs, dtype=np.int64))
    missing_planes = all_planes.difference(non_zero_planes)

    for i in missing_planes:
        output_coordinates[num_unique_indices, 0] = i
        num_unique_indices += 1

    output_coordinates = output_coordinates[:num_unique_indices]
    output_values = output_values[:num_unique_indices]

    sorting_indices = np.argsort((output_coordinates * shape_scale).sum(1))
    output_coordinates = np.ascontiguousarray(output_coordinates[sorting_indices])
    output_values = np.ascontiguousarray(output_values[sorting_indices])
    
    return output_coordinates, output_values

def compress_first_index(indices, values, shape):
    assert np.all(np.diff(indices[:, 0]) >= 0)

    starts = np.searchsorted(indices[:, 0], np.arange(shape[0]))
    ends = np.concatenate((starts[1:], [indices.shape[0]]))

    return np.stack([starts, ends]).T

In [3]:
RENAME_PRONG_TARGET = False

ENERGY_LIMIT = None

APPLY_PRECUT = True

#PATH = "/exp/dune/data/users/linyan/nnbar/h5/step4/atm_nnbar_step4_2M100k6.h5"
#OUTPUT = "/exp/dune/data/users/linyan/nnbar/h5/combined/atm_nnbar_combined_2M100k6.h5"
PATH = "/exp/dune/data/users/linyan/detsum/H5/test.h5"
OUTPUT = "/exp/dune/data/users/linyan/detsum/Sparse/test.h5"

In [4]:
file = h5py.File(PATH, 'r')

num_events, num_prongs, *_ = file["png_cvnmap_shape"][:]

data = file["input_png3d"][:] if "input_png3d" in file else np.zeros((num_events, 24, num_prongs), dtype=np.float32)
mask = png_cvnmap_pad_mask = file['input_png3d_pad_mask'][:] if "input_png3d_pad_mask" in file else np.zeros((num_events, num_prongs), dtype=bool)
extra = file["input_slice"][:] if "input_slice" in file else np.zeros(num_events, dtype=np.float32)
truesE = file["trueE"][:] if "trueE" in file else np.zeros(num_events, dtype=np.float32)
genies_nmult = file["genie_nmult"][:] if "genie_nmult" in file else np.zeros(num_events, dtype=np.int8)
genies_np = file["genie_np"][:] if "genie_np" in file else np.zeros(num_events, dtype=np.int8)
genies_npi0 = file["genie_npi0"][:] if "genie_npi0" in file else np.zeros(num_events, dtype=np.int8)
genies_npic = file["genie_npic"][:] if "genie_npic" in file else np.zeros(num_events, dtype=np.int8)
genies_sum_px = file["genie_sum_px"][:] if "genie_sum_px" in file else np.zeros(num_events, dtype=np.float32)
genies_sum_py = file["genie_sum_py"][:] if "genie_sum_py" in file else np.zeros(num_events, dtype=np.float32)
genies_sum_pz = file["genie_sum_pz"][:] if "genie_sum_pz" in file else np.zeros(num_events, dtype=np.float32)
genies_sum_ke = file["genie_sum_ke"][:] if "genie_sum_ke" in file else np.zeros(num_events, dtype=np.float32)
genies_sphericity = file["genie_sphericity"][:] if "genie_sphericity" in file else np.zeros(num_events, dtype=np.float32)
genies_aplanarity = file["genie_aplanarity"][:] if "genie_aplanarity" in file else np.zeros(num_events, dtype=np.float32)
genies_file_name = (
    np.char.decode(file["file_name"][:], "utf-8")
    if "file_name" in file
    else np.array([""] * num_events, dtype=f"<U{FNAME_ITEMSIZE}")
)
models = file["model"][:] if "model" in file else np.zeros(num_events, dtype=np.int8)
prescut = file["precut"][:] if "precut" in file else np.zeros(num_events, dtype=np.float32)

target = file["mc.inter"][:]
prong_targets = file["mc.png_label_split_photons"][:]

print(f"Shape of 'data': {data.shape}")
print(f"Shape of 'extra': {extra.shape}")
print(f"Shape of 'truesE': {truesE.shape}")
print(f"Shape of 'genies_sphericity': {genies_sphericity.shape}")
print(f"Shape of 'genies_file_name': {genies_file_name.shape}")
print(f"Total elements in 'extra': {extra.size}")

data = data.astype(np.float32)
data = data.transpose((0, 2, 1))
data = np.ascontiguousarray(data)

print(models)
print(genies_sphericity)
#print(genies_file_name)

FNAME_ITEMSIZE = 256

extra = extra.reshape(data.shape[0], -1).astype(np.float32)
truesE = truesE.reshape(data.shape[0], -1).astype(np.float32)
genies_nmult = genies_nmult.reshape(data.shape[0], -1).astype(np.int8)
genies_np = genies_np.reshape(data.shape[0], -1).astype(np.int8)
genies_npi0 = genies_npi0.reshape(data.shape[0], -1).astype(np.int8)
genies_npic = genies_npic.reshape(data.shape[0], -1).astype(np.int8)
genies_sum_px = genies_sum_px.reshape(data.shape[0], -1).astype(np.float32)
genies_sum_py = genies_sum_py.reshape(data.shape[0], -1).astype(np.float32)
genies_sum_pz = genies_sum_pz.reshape(data.shape[0], -1).astype(np.float32)
genies_sum_ke = genies_sum_ke.reshape(data.shape[0], -1).astype(np.float32)
genies_sphericity = genies_sphericity.reshape(data.shape[0], -1).astype(np.float32)
genies_aplanarity = genies_aplanarity.reshape(data.shape[0], -1).astype(np.float32)
genies_file_name = genies_file_name.reshape(data.shape[0], -1).astype(f"<U{FNAME_ITEMSIZE}")
models = models.reshape(data.shape[0], -1).astype(np.int8)
prescut = prescut.reshape(data.shape[0], -1).astype(np.float32)

#unique_targets = np.unique(target)
#target_map = {key: value for key, value in map(reversed, enumerate(unique_targets))}
#target = np.fromiter((target_map[t] for t in target), dtype=np.int64, count=len(target))

current_target = np.zeros_like(target)
current_target[(target > 3) & (target <= 7)] = 1
current_target[target == 8] = 2
current_target[target == 9] = 3

if RENAME_PRONG_TARGET:
    prong_targets[prong_targets == 9] = 8
    
if not mask.any():
    mask = prong_targets >= 0

Shape of 'data': (18954, 8, 20)
Shape of 'extra': (18954, 4)
Shape of 'truesE': (18954,)
Shape of 'genies_sphericity': (18954,)
Shape of 'genies_file_name': (18954,)
Total elements in 'extra': 75816
[4 4 4 ... 4 4 4]
[0.65156662 0.85642018 0.77176232 ... 0.78677375 0.82061336 0.58001864]


In [5]:
del current_target
print("\n".join(file.keys()))
list(zip(range(9), np.bincount(prong_targets[prong_targets >= 0]) / (prong_targets >= 0).sum()))

cvn0neutrons
cvn0pions
cvn0pizeros
cvn0protons
cvn1neutrons
cvn1pions
cvn1pizeros
cvn1protons
cvn2neutrons
cvn2pions
cvn2pizeros
cvn2protons
cvnNneutrons
cvnNpions
cvnNpizeros
cvnNprotons
cvnmap_index
cvnmap_shape
cvnmap_value
cvnnc
cvnnue
cvnnumu
cvnnutau
file_name
genie_aplanarity
genie_nmult
genie_np
genie_npi0
genie_npic
genie_sphericity
genie_sum_ke
genie_sum_px
genie_sum_py
genie_sum_pz
input_png3d
input_png3d_pad_mask
input_slice
isNonFlux
mc.inter
mc.png_label
mc.png_label_split_photons
mc.png_mother
model
png_ShwTrk
png_cvnmap_index
png_cvnmap_shape
png_cvnmap_value
png_trueE
png_trueP
precut
tpc_id
trueE
trueP
trueVertex


[(0, 0.029638598818466474),
 (1, 0.0056531597538214745),
 (2, 0.3772963551387454),
 (3, 0.0005321528281223488),
 (4, 0.2292730329626259),
 (5, 0.34760376980148383),
 (6, 0.006054202464870201),
 (7, 0.003948728231864386)]

In [6]:
png_cvnmap_index = file["png_cvnmap_index"][:]
png_cvnmap_value = file["png_cvnmap_value"][:]
png_cvnmap_shape = file["png_cvnmap_shape"][:]
png_cvnmap_index[:,0]-=png_cvnmap_index[0][0]
cvnmap_index = file["cvnmap_index"][:]
cvnmap_value = file["cvnmap_value"][:]
cvnmap_shape = file["cvnmap_shape"][:]
cvnmap_index[:,0]-=cvnmap_index[0][0]

cvnmap_index.shape

cvnmap_shape

array([18954,     3,   350,   350])

In [7]:
extra_index = np.zeros(cvnmap_index.shape[0], dtype=cvnmap_index.dtype)
cvnmap_index = np.insert(cvnmap_index, 1, extra_index, axis=1)
cvnmap_shape = np.insert(cvnmap_shape, 1, 1)
cvnmap_shape

array([18954,     1,     3,   350,   350])

In [8]:
(extra.ravel() < 1.5).mean()

arr = np.asarray(models).ravel()                 # (N,)
label_min = int(arr.min())
idx = (arr - label_min).astype(np.int64, copy=False)  # (N,)
n_bins = int(idx.max()) + 1 if idx.size else 0
print(n_bins, label_min)

seen_counts = np.bincount(idx, minlength=n_bins)

print(seen_counts)
print(cvnmap_value.shape)
print(cvnmap_index.shape)
print(cvnmap_index)
print(prescut.shape)

cvnmap_index = np.ascontiguousarray(cvnmap_index.T)
cvnmap = sparse.COO(cvnmap_index, cvnmap_value, tuple(map(int, cvnmap_shape)))
cvnmap_index = np.ascontiguousarray(cvnmap.coords.T)
print(cvnmap_index.shape)
print(cvnmap_index)

1 4
[18954]
(12021430,)
(12021430, 5)
[[    0     0     0     0     0]
 [    0     0     0   104   183]
 [    0     0     0   104   184]
 ...
 [18953     0     2   145   175]
 [18953     0     2   145   177]
 [18953     0     2   145   183]]
(18954, 3)
(12021430, 5)
[[    0     0     0     0     0]
 [    0     0     0   104   183]
 [    0     0     0   104   184]
 ...
 [18953     0     2   145   175]
 [18953     0     2   145   177]
 [18953     0     2   145   183]]


In [9]:
if APPLY_PRECUT:
    if prescut.shape[1] != 3:
        raise ValueError(f"prescut must have 3 columns, got {prescut.shape[1]}")
    n_tracks       = prescut[:, 0].astype(np.int32)
    n_showers      = prescut[:, 1].astype(np.int32)
    hit_tot_energy = prescut[:, 2].astype(np.float32)
    # energy_mask = (max(n_tracks, NShw) > 1) && (hit_tot_energy < 2)
    precut_mask = (np.maximum(n_tracks, n_showers) > 2) & (hit_tot_energy < 2.0)
    print(n_tracks, n_showers, hit_tot_energy, models, precut_mask)

    extra = extra[precut_mask]
    data = data[precut_mask]
    mask = mask[precut_mask]
    truesE = truesE[precut_mask]
    genies_nmult = genies_nmult[precut_mask]
    genies_np = genies_np[precut_mask]
    genies_npi0 = genies_npi0[precut_mask]
    genies_npic = genies_npic[precut_mask]
    genies_sum_px = genies_sum_px[precut_mask]
    genies_sum_py = genies_sum_py[precut_mask]
    genies_sum_pz = genies_sum_pz[precut_mask]
    genies_sum_ke = genies_sum_ke[precut_mask]
    genies_sphericity = genies_sphericity[precut_mask]
    genies_aplanarity = genies_aplanarity[precut_mask]
    genies_file_name = genies_file_name[precut_mask]
    models = models[precut_mask]
    prescut = prescut[precut_mask]
    print("\n reach here\n")

    target = target[precut_mask]
    prong_targets = prong_targets[precut_mask]

    cvnmap_index = np.ascontiguousarray(cvnmap_index.T)
    cvnmap = sparse.COO(cvnmap_index, cvnmap_value, tuple(map(int, cvnmap_shape)))
    cvnmap = cvnmap[precut_mask]
    cvnmap_value = cvnmap.data
    cvnmap_index = np.ascontiguousarray(cvnmap.coords.T)
    cvnmap_shape = np.array(cvnmap.shape)
    
    png_cvnmap_index = np.ascontiguousarray(png_cvnmap_index.T)    
    png_cvnmap = sparse.COO(png_cvnmap_index, png_cvnmap_value, tuple(map(int, png_cvnmap_shape)))    
    png_cvnmap = png_cvnmap[precut_mask]
    png_cvnmap_value = png_cvnmap.data
    png_cvnmap_index = np.ascontiguousarray(png_cvnmap.coords.T)
    png_cvnmap_shape = np.array(png_cvnmap.shape)
    

[8 8 9 ... 7 7 5] [8 8 9 ... 7 7 5] [1.0023713  1.3324671  1.117369   ... 1.4558008  1.0737909  0.62081814] [[4]
 [4]
 [4]
 ...
 [4]
 [4]
 [4]] [ True  True  True ...  True  True  True]

 reach here



In [10]:
pass_counts = np.bincount(idx[precut_mask], minlength=n_bins)

# Map back to original labels
model_ids = np.arange(n_bins, dtype=int) + label_min

print("\nPer-model events passing energy_mask:")
for lab, seen, passed in zip(model_ids, seen_counts, pass_counts):
    rate = (passed / seen) if seen else 0.0
    print(f"{lab}  {int(passed):8d} / {int(seen):8d}  ({rate:6.2%})")



Per-model events passing energy_mask:
4     18543 /    18954  (97.83%)


In [11]:
if os.path.exists(PATH.replace(".h5", ".select.npy")):
    select_mask = np.load(PATH.replace(".h5", ".select.npy")) > 0.5
    select_mask = np.append(select_mask, [False])
    
    png_cvnmap_index = np.ascontiguousarray(png_cvnmap_index.T)
    cvnmap_index = np.ascontiguousarray(cvnmap_index.T)

    png_cvnmap = sparse.COO(png_cvnmap_index, png_cvnmap_value, tuple(map(int, png_cvnmap_shape)))
    cvnmap = sparse.COO(cvnmap_index, cvnmap_value, tuple(map(int, cvnmap_shape)))
    
    cvnmap = cvnmap[select_mask]
    png_cvnmap = png_cvnmap[select_mask]
    
    extra = extra[select_mask]
    data = data[select_mask]
    mask = mask[select_mask]
    
    target = target[select_mask]
    prong_targets = prong_targets[select_mask]
    
    cvnmap_value = cvnmap.data
    cvnmap_index = np.ascontiguousarray(cvnmap.coords.T)
    cvnmap_shape = np.array(cvnmap.shape)

    png_cvnmap_value = png_cvnmap.data
    png_cvnmap_index = np.ascontiguousarray(png_cvnmap.coords.T)
    png_cvnmap_shape = np.array(png_cvnmap.shape)

In [12]:
event_compressed_index = compress_first_index(cvnmap_index, cvnmap_value, cvnmap_shape)
prong_compressed_index = compress_first_index(png_cvnmap_index, png_cvnmap_value, png_cvnmap_shape)

image_size = cvnmap_shape[-2:]
shape_scale = np.ascontiguousarray(np.cumprod((*cvnmap_shape[-2:], 1)[::-1])[::-1])

image_size = png_cvnmap_shape[-2:]
shape_scale = np.ascontiguousarray(np.cumprod((*png_cvnmap_shape[-2:], 1)[::-1])[::-1])

In [13]:
from joblib import Parallel, delayed

In [14]:
def extract_event(i):
    lower, upper = event_compressed_index[i]
    num_prongs = 1
    num_features = cvnmap_shape[2]

    event_coordinates = cvnmap_index[lower:upper, 1:]
    event_values = cvnmap_value[lower:upper]
    return sparse_to_sparse(event_coordinates, event_values, num_prongs, num_features, shape_scale)

In [15]:
event_results = Parallel(n_jobs=32, verbose=1)(delayed(extract_event)(i) for i in range(len(event_compressed_index)))
all_event_coordinates = [result[0] for result in event_results]
all_event_values = [result[1] for result in event_results]

[Parallel(n_jobs=32)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done 164 tasks      | elapsed:   10.8s
[Parallel(n_jobs=32)]: Done 2112 tasks      | elapsed:   11.3s
[Parallel(n_jobs=32)]: Done 18543 out of 18543 | elapsed:   13.0s finished


In [16]:
del cvnmap_value, cvnmap_index, event_results

In [17]:
event_lengths = np.array([0] + list(map(len, all_event_coordinates)))
event_compressed_index = np.stack((np.cumsum(event_lengths)[:-1], np.cumsum(event_lengths)[1:]), axis=1)

all_event_values = np.concatenate(all_event_values)
all_event_coordinates = np.concatenate(all_event_coordinates)

In [18]:
del event_lengths

In [19]:
def extract_prong(i):
    lower, upper = prong_compressed_index[i]
    
    num_prongs = max(mask[i].sum(), 1)
    num_features = png_cvnmap_shape[2]
    
    prong_coordinates = png_cvnmap_index[lower:upper, 1:]
    prong_values = png_cvnmap_value[lower:upper]
    return sparse_to_sparse(prong_coordinates, prong_values, num_prongs, num_features, shape_scale)

In [20]:
prong_results = Parallel(n_jobs=32, verbose=1)(delayed(extract_prong)(i) for i in range(len(prong_compressed_index)))
all_prong_coordinates = [result[0] for result in prong_results]
all_prong_values = [result[1] for result in prong_results]

[Parallel(n_jobs=32)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=32)]: Done 224 tasks      | elapsed:    0.5s
[Parallel(n_jobs=32)]: Done 4160 tasks      | elapsed:    1.2s
[Parallel(n_jobs=32)]: Done 18543 out of 18543 | elapsed:    2.7s finished


In [21]:
del png_cvnmap_value, png_cvnmap_index, prong_results

In [22]:
prong_lengths = np.array([0] + list(map(len, all_prong_coordinates)))
prong_compressed_index = np.stack((np.cumsum(prong_lengths)[:-1], np.cumsum(prong_lengths)[1:]), axis=1)

all_prong_values = np.concatenate(all_prong_values)
all_prong_coordinates = np.concatenate(all_prong_coordinates)

In [23]:
del prong_lengths

In [24]:
all_event_coordinates = all_event_coordinates.astype(np.int32)
all_prong_coordinates = all_prong_coordinates.astype(np.int32)
num_events, max_prongs, *full_pixels_shape = png_cvnmap_shape

In [25]:
OUTPUT_FILE = f"{OUTPUT}"
if ENERGY_LIMIT:
    OUTPUT_FILE = f"{OUTPUT}_{ENERGY_LIMIT}_gev.h5"
    
print(OUTPUT_FILE)

/exp/dune/data/users/linyan/detsum/Sparse/test.h5


In [26]:
with h5py.File(OUTPUT_FILE, 'w') as output_file:
    output_file.create_dataset("features", data=data)
    output_file.create_dataset("extra", data=extra)
    output_file.create_dataset("truesE", data=truesE)
    output_file.create_dataset("genie_nmult", data=genies_nmult)
    output_file.create_dataset("genie_np", data=genies_np)
    output_file.create_dataset("genie_npi0", data=genies_npi0)
    output_file.create_dataset("genie_npic", data=genies_npic)
    output_file.create_dataset("genie_sum_px", data=genies_sum_px)
    output_file.create_dataset("genie_sum_py", data=genies_sum_py)
    output_file.create_dataset("genie_sum_pz", data=genies_sum_pz)
    output_file.create_dataset("genie_sum_ke", data=genies_sum_ke)
    output_file.create_dataset("genie_sphericity", data=genies_sphericity)
    output_file.create_dataset("genie_aplanarity", data=genies_aplanarity)
    genies_file_name = (
    np.char.encode(genies_file_name, "utf-8")
    .reshape(data.shape[0], -1)
    .astype(f"S{FNAME_ITEMSIZE}")
    )
    output_file.create_dataset("file_name", data=genies_file_name)
    output_file.create_dataset("prescut", data=prescut)
    output_file.create_dataset("models", data=models) 

In [27]:
unique_targets_last = np.unique(target)
print(f"Original event targets ('target'): {unique_targets_last}")
print(f"Number of original event targets: {len(unique_targets_last)}")
print("-" * 30)
print(f"Shape of 'data': {data.shape}")
print(f"Shape of 'extra': {extra.shape}")
print(f"Shape of 'target': {target.shape}")
print(f"Shape of 'genies_sphericity': {genies_sphericity.shape}")
print(f"Shape of 'genies_file_name': {genies_file_name.shape}")
print(f"Total elements in 'extra': {extra.size}")

with h5py.File(OUTPUT_FILE, 'a') as output_file:
    output_file.create_dataset("event_target", data=target)
    output_file.create_dataset("event_pixels_coordinates", data=all_event_coordinates)
    output_file.create_dataset("event_pixels_values", data=all_event_values)
    output_file.create_dataset("event_pixels_shape", data=cvnmap_shape[1:])
    output_file.create_dataset("event_compressed_index", data=event_compressed_index)

Original event targets ('target'): [1000180416]
Number of original event targets: 1
------------------------------
Shape of 'data': (18543, 20, 8)
Shape of 'extra': (18543, 4)
Shape of 'target': (18543,)
Shape of 'genies_sphericity': (18543, 1)
Shape of 'genies_file_name': (18543, 1)
Total elements in 'extra': 74172


In [28]:
del all_event_coordinates,all_event_values,event_compressed_index

In [29]:
with h5py.File(OUTPUT_FILE, 'a') as output_file:
    output_file.create_dataset("prong_target", data=prong_targets)
    output_file.create_dataset("prong_mask", data=mask)
    
    output_file.create_dataset("prong_pixels_coordinates", data=all_prong_coordinates)
    output_file.create_dataset("prong_pixels_values", data=all_prong_values)
    output_file.create_dataset("prong_pixels_shape", data=png_cvnmap_shape[1:])
    output_file.create_dataset("prong_compressed_index", data=prong_compressed_index)

    output_file.create_dataset("full_pixels_shape", data=np.array(full_pixels_shape))